<center><h1>Zhan_Jenna_HW3</h1></center>
<br>
<br>

Name: Jenna Zhan
<br>
Github Username: jenna-zhan
<br>
USC ID: 2477604908

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
import os
import pandas as pd
import numpy as np

Get the AReM Data Set

In [2]:
arem_path = "../data/AReM"
arem = os.listdir(arem_path)
print(arem)

['bendingType.pdf', '.DS_Store', 'bending1', 'walking', 'sensorsPlacement.pdf', 'bending2', 'standing', 'sitting', 'lying', 'cycling']


### (b) Test and Train Data

In [3]:
activities = [
    "bending1",
    "bending2",
    "cycling",
    "lying",
    "sitting",
    "standing",
    "walking"
]

train_set = []
test_set = []

for activity in activities:
    activity_path = os.path.join(arem_path, activity)
    dataset = os.listdir(activity_path)

    print(activity, len(dataset))

bending1 7
bending2 6
cycling 15
lying 15
sitting 15
standing 15
walking 15


In [4]:
for activity in activities:
    activity_path = os.path.join(arem_path, activity)
    
    dataset = [
        file for file in os.listdir(activity_path)
        if file.startswith("dataset") and file.endswith(".csv")
    ]

    for file in dataset:
        file_num = int(
            file.replace("dataset", "").replace(".csv", "")
        )

        file_path = os.path.join(activity_path, file)

        if activity in ["bending1", "bending2"]:
            if file_num in [1, 2]:
                test_set.append(file_path)
            else:
                train_set.append(file_path)
        else:
            if file_num in [1, 2, 3]:
                test_set.append(file_path)
            else:
                train_set.append(file_path)

print("training instances:", len(train_set))
print("test instances:", len(test_set))
print("total instances:", len(train_set) + len(test_set))

training instances: 69
test instances: 19
total instances: 88


### (c) Feature Extraction

#### i. Research

Common time-domain features include: mean, variance, mode, median, skewness (third moment describing data asymmetry), kurtosis (fourth moment determining tailedness of the distribution), minimum, maximum, coefficient of variation (CV), first quartile (Q1), third quartile (Q3), interquartile range (IQR), energy, average power, root mean squared value (RMS), and line length (curve length).

Reference: Poomipat Boonyakitanont, Apiwat Lek-uthai, Krisnachai Chomtho, Jitkomut Songsiri,
A review of feature extraction and performance evaluation in epileptic seizure detection using EEG,
Biomedical Signal Processing and Control,
Volume 57,
2020,
101702,
ISSN 1746-8094,
https://doi.org/10.1016/j.bspc.2019.101702.
(https://www.sciencedirect.com/science/article/pii/S1746809419302836)

#### ii. Extraction

In [5]:
time_series = [
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23"
]

In [6]:
all_files = train_set + test_set

print(len(all_files))

88


In [7]:
def extract_features(file_path):

    if "bending2/dataset4.csv" in file_path:
        data = pd.read_csv(
            file_path,
            skiprows=5,
            sep=r"\s+",
            names=[
                "time",
                "avg_rss12",
                "var_rss12",
                "avg_rss13",
                "var_rss13",
                "avg_rss23",
                "var_rss23"
            ]
        )
    else:
        data = pd.read_csv(
            file_path,
            skiprows=4,
            usecols=range(7)
        )

    features = {}

    for i, column in enumerate(time_series, start=1):
        ts = data[column]

        features[f"min{i}"] = ts.min()
        features[f"max{i}"] = ts.max()
        features[f"mean{i}"] = ts.mean()
        features[f"median{i}"] = ts.median()
        features[f"std{i}"] = ts.std()
        features[f"1st_quart{i}"] = ts.quantile(0.25)
        features[f"3rd_quart{i}"] = ts.quantile(0.75)

    return features

In [8]:
for file in all_files:
    try:
        extract_features(file)
    except Exception as e:
        print("problem file:", file)
        print("error:", e)

In [9]:
problem_files = [
    "../data/AReM/cycling/dataset14.csv",
    "../data/AReM/cycling/dataset9.csv"
]

for file in problem_files:
    print("\n", file)

    with open(file, "r") as f:
        lines = f.readlines()

    for line in lines[-5:]:
        print(line.strip())


 ../data/AReM/cycling/dataset14.csv
118750,35.00,3.00,21.50,2.50,15.75,7.76
119000,40.25,2.49,19.75,3.27,21.50,2.29
119250,41.50,0.87,12.75,4.44,17.75,2.38
119500,33.33,6.18,15.50,3.77,17.00,4.32
119750,34.00,2.12,24.50,1.80,22.25,2.59,

 ../data/AReM/cycling/dataset9.csv
118750,41.75,1.79,11.00,6.00,16.67,2.49
119000,36.33,0.47,16.00,3.16,20.33,1.70
119250,31.50,1.50,21.00,0.00,12.25,7.12
119500,34.25,6.38,12.67,2.49,15.25,4.21
119750,38.33,0.94,15.25,2.17,20.33,1.25,


In [10]:
all_features = []

for file in all_files:
    features = extract_features(file)
    all_features.append(features)

feature_df = pd.DataFrame(all_features)

print(feature_df.shape)
feature_df.head()

(88, 42)


,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,mean2,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
0,36.25,48.00,43.969125,44.50,1.618364,43.31,44.67,0.0,1.50,0.413125,...,3.318301,20.5000,23.75,0.0,2.96,0.555312,0.49,0.487826,0.00,0.83
1,37.00,48.00,43.454958,43.25,1.386098,42.50,45.00,0.0,1.58,0.378083,...,2.488862,22.2500,24.00,0.0,5.26,0.679646,0.50,0.622534,0.43,0.87
2,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,3.00,0.696042,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.00,1.00
3,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,2.83,0.535979,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.00,0.50
4,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,0.0,1.70,0.426250,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.00,0.94


In [11]:
print(feature_df.isna().sum().sum())

0


In [12]:
feature_table = feature_df.copy()
feature_table.insert(0, "instance", range(1, 89))

feature_table

,instance,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
0,1,36.25,48.00,43.969125,44.500,1.618364,43.31,44.6700,0.0,1.50,...,3.318301,20.5000,23.750,0.0,2.96,0.555312,0.490,0.487826,0.000,0.830
1,2,37.00,48.00,43.454958,43.250,1.386098,42.50,45.0000,0.0,1.58,...,2.488862,22.2500,24.000,0.0,5.26,0.679646,0.500,0.622534,0.430,0.870
2,3,33.00,47.75,42.179812,43.500,3.670666,39.15,45.0000,0.0,3.00,...,3.849448,30.4575,36.330,0.0,2.18,0.613521,0.500,0.524317,0.000,1.000
3,4,33.00,45.75,41.678063,41.750,2.243490,41.33,42.7500,0.0,2.83,...,2.411026,28.4575,31.250,0.0,1.79,0.383292,0.430,0.389164,0.000,0.500
4,5,35.00,47.40,43.954500,44.330,1.558835,43.00,45.0000,0.0,1.70,...,1.999604,35.3625,36.500,0.0,1.79,0.493292,0.430,0.513506,0.000,0.940
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,84,35.50,46.25,43.174938,43.670,1.989052,42.50,44.5000,0.0,2.12,...,2.983976,12.7500,16.500,0.0,5.72,0.911979,0.830,0.666161,0.470,1.220
84,85,32.75,47.00,42.760562,44.500,3.398919,41.33,45.3725,0.0,3.34,...,4.296574,13.0000,18.565,0.0,5.73,0.842271,0.710,0.722165,0.430,1.090
85,86,19.33,43.50,34.227771,35.500,4.889576,30.50,37.7500,0.0,14.50,...,3.092094,14.7500,18.670,0.0,9.74,3.394125,3.100,1.792090,2.105,4.425
86,87,12.50,45.00,33.509729,34.125,4.850923,30.50,36.7500,0.0,13.05,...,3.133564,14.6275,18.750,0.0,8.96,3.378479,3.085,1.787360,2.060,4.440


#### iii. Standard Deviation

In [13]:
feature_std = feature_df.std()

feature_std

min1          9.569975
max1          4.394362
mean1         5.335718
median1       5.440054
std1          1.772153
1st_quart1    6.153590
3rd_quart1    5.138925
min2          0.000000
max2          5.062729
mean2         1.574164
median2       1.412244
std2          0.884105
1st_quart2    0.946386
3rd_quart2    2.125266
min3          2.956462
max3          4.875137
mean3         4.008380
median3       4.036396
std3          0.946710
1st_quart3    4.220658
3rd_quart3    4.171628
min4          0.000000
max4          2.183625
mean4         1.166114
median4       1.145586
std4          0.458242
1st_quart4    0.843620
3rd_quart4    1.552504
min5          6.124001
max5          5.741238
mean5         5.675593
median5       5.813782
std5          1.024898
1st_quart5    6.096465
3rd_quart5    5.531720
min6          0.045838
max6          2.518921
mean6         1.154812
median6       1.086474
std6          0.517617
1st_quart6    0.758584
3rd_quart6    1.523599
dtype: float64

In [14]:
feature_df.isna().sum()[feature_df.isna().sum() > 0]

feature_df[feature_df.isna().any(axis=1)]

nan_rows = feature_df[feature_df.isna().any(axis=1)].index

for i in nan_rows:
    print("row:", i)
    print("file:", all_files[i])


In [15]:
problem_file = "../data/AReM/bending2/dataset4.csv"

with open(problem_file, "r") as f:
    lines = f.readlines()

for line in lines[:10]:
    print(line.strip())

# Task: bending2
# Frequency (Hz): 20
# Clock (millisecond): 250
# Duration (seconds): 120
# Columns: time,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23
0 32.50 0.50 0.00 0.00 19.00 1.00
250 32.50 0.50 0.00 0.00 18.50 0.50
500 32.75 0.43 1.00 0.00 18.00 0.00
750 32.50 0.50 0.00 0.00 17.50 0.50
1000 32.50 0.50 7.50 0.50 17.50 0.87


In [19]:
np.random.seed(5)

bootstrap_results = []

for column in feature_df.columns:
    values = feature_df[column].values
    bootstrap_std = []

    for i in range(1000):
        bootstrap_sample = np.random.choice(
            values,
            size=len(values),
            replace=True
        )

        bootstrap_std.append(
            np.std(bootstrap_sample, ddof=1)
        )

    ci_lower = np.percentile(bootstrap_std, 5)
    ci_upper = np.percentile(bootstrap_std, 95)

    bootstrap_results.append({
        "feature": column,
        "standard deviation": feature_std[column],
        "90% ci lower": ci_lower,
        "90% ci upper": ci_upper
    })

bootstrap_df = pd.DataFrame(bootstrap_results)

bootstrap_df

,feature,standard deviation,90% ci lower,90% ci upper
0,min1,9.569975,8.204965,10.709391
1,max1,4.394362,3.403882,5.300683
2,mean1,5.335718,4.711753,5.892600
3,median1,5.440054,4.777375,5.988009
4,std1,1.772153,1.571347,1.935718
5,1st_quart1,6.153590,5.591167,6.629550
6,3rd_quart1,5.138925,4.344417,5.865224
7,min2,0.000000,0.000000,0.000000
8,max2,5.062729,4.626905,5.389798
9,mean2,1.574164,1.400806,1.709856


#### iv. Select Features

In [18]:
feature_types = [
    "min",
    "max",
    "mean",
    "median",
    "std",
    "1st_quart",
    "3rd_quart"
]

feature_comparison = {}

for feature in feature_types:
    columns = [f"{feature}{i}" for i in range(1, 7)]
    feature_comparison[feature] = feature_df[columns].std().mean()

feature_comparison = pd.Series(
    feature_comparison
).sort_values(ascending=False)

feature_comparison

max          4.129335
3rd_quart    3.340607
1st_quart    3.169884
median       3.155756
mean         3.152463
min          3.116046
std          0.933954
dtype: float64

I select maximum, third quartile, and mean as the three most important time-domain features. Features with larger standard deviations vary more across instances and may therefore carry more information for distinguishing between instances. Maximum and third quartile have the two largest average standard deviations. Although first quartile ranks next, I do not select it because it is similar to the third quartile and may provide redundant information. Mean and median have very similar average standard deviations, so I select mean because it provides a more direct summary of the overall level of the time series.

## 2. ISLR 3.7.4

### (a) Linear Train

The linear model is a special case of the cubic model when the coefficients of the quadratic and cubic terms are zero. Since the cubic model has two additional terms, it is more flexible and can fit the training data at least as well as the linear model. Therefore, the cubic model is expected to have a lower or equal training RSS.

### (b) Linear Test

Since the true relationship is linear, the linear model is expected to perform better on new test data. The cubic model is more flexible and may fit some noise or characteristics specific to the training data, which can lead to overfitting. Therefore, the linear model is expected to have a lower test RSS than the cubic model.

### (c) Not Linear Train

Since the true relationship is not linear, the cubic model is expected to have a lower training RSS. It is more flexible because it includes additional quadratic and cubic terms, which allow it to capture more complex non-linear patterns in the data.Even though we do not know how far the true relationship is from linear, the cubic model can still reduce to a linear model by setting the coefficients of the quadratic and cubic terms equal to zero. Therefore, the cubic model is at least as flexible as the linear model.

### (d) Not Linear Testing

There is not enough information to tell because we do not know how far the true relationship between X and Y is from linear. If the true relationship is very close to linear, the cubic model may overfit the training data, and the linear model may have a lower test RSS. On the other hand, if the true relationship is close to cubic, the cubic model may capture the non-linear relationship better and have a lower test RSS. Therefore, we cannot determine which model will have a lower test RSS without more information about the true relationship.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 